In [1]:
import os

try:
    import google.colab
    IS_COLAB = True
    print("Google Colab")
except ImportError:
    IS_COLAB = False
    print("로컬 Jupyter")

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPEN_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("import 완료")
print("API 키 OK" if os.environ.get("OPENAI_API_KEY") else "키 없음")

로컬 Jupyter
import 완료
API 키 OK


In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import json

# personas.json 파일을 읽기 모드로 열고
# JSON 데이터를 Python 딕셔너리로 변환
with open("personas.json", encoding="utf-8") as f:
    PERSONAS = json.load(f)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "{persona}"),
    ("human", "{question}")
])


# 프롬프트와 LLM 연결
chain = prompt | llm


def ask(persona_name: str, question: str) -> str:
    system_message = PERSONAS[persona_name]

    # 체인 실행
    result = chain.invoke({
        "persona": system_message,
        "question": question
    })

    # AIMessage에서 답변 내용만 반환
    return result.content

def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)

    # 첫 번째 값을 명령어로 사용
    cmd = parts[0]

    # 종료 명령어
    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True

    # 현재 페르소나 확인
    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    # 도움말
    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")

    # 페르소나 변경
    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""

        # JSON에서 불러온 PERSONAS에 존재하는지 확인
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False

        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")

            # PERSONAS의 key들을 출력
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    # 등록되지 않은 명령어
    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

def run_chatbot(initial_persona: str = "분석가"):

    # 처음 사용할 페르소나
    persona = initial_persona

    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:

        # 사용자 입력
        user_input = input("질문> ")
        user_input = user_input.strip()

        # 빈 입력이면 다시 입력받기
        if not user_input:
            continue

        # "/"로 시작하면 명령어로 처리
        if user_input.startswith("/"):

            persona, should_quit = handle_command(
                user_input,
                persona
            )

            # /quit 또는 /exit이면 반복문 종료
            if should_quit:
                break

        # "/"로 시작하지 않으면 일반 질문
        else:

            # 현재 선택된 persona를 사용해서 질문
            answer = ask(
                persona,
                user_input
            )

            print(f"[{persona}] {answer}\n")

In [10]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

❓ 알 수 없는 명령어: /디자이너 (/help 입력)
✅ 페르소나가 '디자이너'(으)로 변경되었습니다.
[디자이너] 안녕하세요! 😊 어떻게 도와드릴까요? 사용자 경험이나 디자인에 관한 질문이 있다면 언제든지 말씀해 주세요!

👋 종료합니다.
